In [ ]:
%load_ext rich

In [ ]:
import json
import os
from pathlib import Path

import requests
from tqdm import tqdm

from aymurai.experiments.entity_disambiguation.runner import (
    call_extraction_api as extract_document,
)

API_URL = "http://localhost:8000"  # Url for debugger. change it to your own
DATA_ROOT = Path(
    os.getenv(
        "DISAMBIGUATION_DATA_ROOT",
        "../../../resources/data/restricted/disambiguation-eval/files",
    )
)
DOC_EXTENSIONS = {".pdf", ".docx"}

## Sample document


In [ ]:
def discover_documents(root: Path, extensions: set[str]) -> list[Path]:
    extensions = {ext.lower() for ext in extensions}
    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in extensions
    )

In [ ]:
documents = discover_documents(DATA_ROOT, DOC_EXTENSIONS)

print(f"Found {len(documents)} documents")

doc_path = documents[1]

## /document-extract endpoint output


In [ ]:
# /document-extract endpoint output
session = requests.Session()
document = extract_document(
    session,
    endpoint=f"{API_URL}/misc/document-extract",
    file_path=doc_path,
    timeout_s=300,
)
paragraphs = document["detail"]["document"]
document

In [ ]:
paragraphs

In [ ]:
len(paragraphs)

## Inference


In [ ]:
# Function to make single inference using the API
def get_predictions(sample: str) -> dict:
    response = requests.post(url=f"{API_URL}/anonymizer/predict", json={"text": sample}, params={"use_cache": False})
    response.raise_for_status()
    return response.json()

In [ ]:
# Check /anonymizer/predict-batch endpoint with progress and timing
from time import perf_counter

CLIENT_BATCH_SIZE = 2  # Number of paragraphs per HTTP request

predictions_batch = []
start_t = perf_counter()
total = len(paragraphs)

for i in tqdm(range(0, total, CLIENT_BATCH_SIZE), desc="Batch predict", unit="chunk"):
    chunk = paragraphs[i : i + CLIENT_BATCH_SIZE]
    batch_payload = [{"text": paragraph} for paragraph in chunk]
    response_batch = requests.post(
        url=f"{API_URL}/anonymizer/predict-batch",
        json=batch_payload,
        params={"use_cache": False},
    )
    response_batch.raise_for_status()

    chunk_predictions = response_batch.json()["data"]
    predictions_batch.extend(chunk_predictions)

elapsed = perf_counter() - start_t
processed = len(predictions_batch)
print(f"Processed paragraphs: {processed}/{total}")
print(f"Total time: {elapsed:.2f}s")
print(f"Avg per paragraph: {elapsed / processed:.4f}s" if processed else "No paragraphs processed")
assert processed == total
predictions_batch[0] if predictions_batch else {}


In [ ]:
predictions = [get_predictions(paragraph) for paragraph in tqdm(paragraphs)]
predictions

## Export variants


In [ ]:
def disambiguate_and_export(
    variant_name: str, label_policies: dict, render_policy: dict
):
    response = requests.post(
        url=f"{API_URL}/anonymizer/disambiguate",
        json={
            "paragraphs": predictions,
            # "custom_prompts": {"root": []},
            "label_policies": label_policies,
        },
    )
    response.raise_for_status()
    disambiguated = response.json()

    json_prediction = json.dumps(
        {
            "data": disambiguated["data"],
            "label_policies": label_policies,
            "render_policy": render_policy,
        },
        indent=2,
        ensure_ascii=False,
    )

    with open(doc_path, "rb") as file:
        files = {"file": file}

        response = requests.post(
            url=f"{API_URL}/anonymizer/anonymize-document",
            data={"annotations": json_prediction},
            files=files,
        )
        response.raise_for_status()

    output_dir = "output"
    os.makedirs(output_dir, exist_ok=True)

    json_output_dir = Path("../../../resources/outputs/entity-disambiguation")
    json_output_dir.mkdir(parents=True, exist_ok=True)

    filename = os.path.basename(doc_path)
    filename, ext = os.path.splitext(filename)
    out_path = f"{output_dir}/{filename}-{variant_name}.odt"
    json_out_path = json_output_dir / f"{filename}-{variant_name}.json"

    with open(out_path, "wb") as file:
        file.write(response.content)

    json_out_path.write_text(json_prediction, encoding="utf-8")

    return json_prediction, out_path, str(json_out_path)

In [ ]:
render_policy = {
    "suffix_mode": "auto",
    "suffix_threshold": 1,
}

# 1) everything fuzzy
label_policies_all_fuzzy = {
    "PER": {"disambiguation": "fuzzy", "anonymize": True, "use_subclass_when_available": True},
    "DNI": {"disambiguation": "fuzzy", "anonymize": True, "use_subclass_when_available": False},
    "LOC": {"disambiguation": "fuzzy", "anonymize": True, "use_subclass_when_available": False},
    "DIRECCION": {"disambiguation": "fuzzy", "anonymize": True, "use_subclass_when_available": False},
    "FECHA": {"disambiguation": "fuzzy", "anonymize": True, "use_subclass_when_available": True},
}
out_all_fuzzy_json, out_all_fuzzy, out_all_fuzzy_json_path = disambiguate_and_export(
    "all-fuzzy", label_policies_all_fuzzy, render_policy
)

# 2) FECHAs excluded
label_policies_no_fecha = {
    "PER": {"disambiguation": "fuzzy", "anonymize": True, "use_subclass_when_available": True},
    "DNI": {"disambiguation": "fuzzy", "anonymize": True, "use_subclass_when_available": False},
    "LOC": {"disambiguation": "fuzzy", "anonymize": True, "use_subclass_when_available": False},
    "DIRECCION": {"disambiguation": "fuzzy", "anonymize": True, "use_subclass_when_available": False},
    "FECHA": {"disambiguation": "none", "anonymize": False, "use_subclass_when_available": True},
}
out_no_fecha_json, out_no_fecha, out_no_fecha_json_path = disambiguate_and_export(
    "no-fecha", label_policies_no_fecha, render_policy
)

# 3) PER llm-disambiguated
label_policies_per_llm = {
    "PER": {"disambiguation": "llm", "anonymize": True, "use_subclass_when_available": True},
    "DNI": {"disambiguation": "fuzzy", "anonymize": True, "use_subclass_when_available": False},
    "LOC": {"disambiguation": "fuzzy", "anonymize": True, "use_subclass_when_available": False},
    "DIRECCION": {"disambiguation": "fuzzy", "anonymize": True, "use_subclass_when_available": False},
    "FECHA": {"disambiguation": "none", "anonymize": True, "use_subclass_when_available": True},
}
out_per_llm_json, out_per_llm, out_per_llm_json_path = disambiguate_and_export("per-llm", label_policies_per_llm, render_policy)

for name, value, path in [
    ("all_fuzzy", out_all_fuzzy_json, out_all_fuzzy_json_path),
    ("no_fecha", out_no_fecha_json, out_no_fecha_json_path),
    ("per_llm", out_per_llm_json, out_per_llm_json_path),
]:
    print(f"\n{name}:")
    print(value)
    print(f"saved to: {path}")